# Feature Engineering V2
Added ELO ratings and player strength scores

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

BASE = r'C:\Project\FIFA_World_Cup_2026'

results = pd.read_csv(os.path.join(BASE, 'Cleaned_Data', 'results_cleaned_v2.csv'))
elo = pd.read_csv(os.path.join(BASE, 'Cleaned_Data', 'elo_cleaned.csv'))
player_strength = pd.read_csv(os.path.join(BASE, 'Cleaned_Data', 'player_strength.csv'))

results['date'] = pd.to_datetime(results['date'])

print("Results:", results.shape)
print("ELO:", elo.shape)
print("Player strength:", player_strength.shape)

Results: (8068, 11)
ELO: (48, 10)
Player strength: (40, 11)


## Rolling Form and Goal Functions

In [2]:
def get_team_form(df, team, date, n=10):
    team_matches = df[
        ((df['home_team'] == team) | (df['away_team'] == team)) &
        (df['date'] < date)
    ].tail(n)
    
    if len(team_matches) == 0:
        return 0.5
    
    wins = 0
    for _, row in team_matches.iterrows():
        if row['home_team'] == team and row['result'] == 1:
            wins += 1
        elif row['away_team'] == team and row['result'] == -1:
            wins += 1
    
    return wins / len(team_matches)

def get_avg_goals(df, team, date, n=10):
    team_matches = df[
        ((df['home_team'] == team) | (df['away_team'] == team)) &
        (df['date'] < date)
    ].tail(n)
    
    if len(team_matches) == 0:
        return 1.0, 1.0
    
    scored, conceded = [], []
    for _, row in team_matches.iterrows():
        if row['home_team'] == team:
            scored.append(row['home_score'])
            conceded.append(row['away_score'])
        else:
            scored.append(row['away_score'])
            conceded.append(row['home_score'])
    
    return np.mean(scored), np.mean(conceded)

print("Functions defined!")

Functions defined!


## Build V2 Feature Matrix

In [ ]:
# Create ELO lookup dict
elo_lookup = elo.set_index('team')['elo_rating'].to_dict()
elo_winrate_lookup = elo.set_index('team')['elo_win_rate'].to_dict()

# Create player strength lookup
ps = player_strength.set_index('team')

def get_player_strength(team, col, default=0):
    try:
        val = ps.loc[team, col]
        return val if not pd.isna(val) else default
    except:
        return default

features = []

for idx, row in results.iterrows():
    home = row['home_team']
    away = row['away_team']
    date = row['date']
    
    home_form = get_team_form(results, home, date)
    away_form = get_team_form(results, away, date)
    
    home_scored, home_conceded = get_avg_goals(results, home, date)
    away_scored, away_conceded = get_avg_goals(results, away, date)
    
    # ELO features
    home_elo = elo_lookup.get(home, 1500)
    away_elo = elo_lookup.get(away, 1500)
    home_elo_wr = elo_winrate_lookup.get(home, 0.5)
    away_elo_wr = elo_winrate_lookup.get(away, 0.5)
    
    # Player strength features
    home_attack = get_player_strength(home, 'attack_goals')
    away_attack = get_player_strength(away, 'attack_goals')
    home_defense = get_player_strength(home, 'def_tackles')
    away_defense = get_player_strength(away, 'def_tackles')
    home_mid = get_player_strength(home, 'mid_tackles')
    away_mid = get_player_strength(away, 'mid_tackles')
    
    is_neutral = 1 if row['neutral'] else 0
    
    features.append({
        'date': date,
        'home_team': home,
        'away_team': away,
        'home_form': home_form,
        'away_form': away_form,
        'form_diff': home_form - away_form,
        'home_avg_scored': home_scored,
        'home_avg_conceded': home_conceded,
        'away_avg_scored': away_scored,
        'away_avg_conceded': away_conceded,
        'goal_diff': home_scored - away_scored,
        'home_elo': home_elo,
        'away_elo': away_elo,
        'elo_diff': home_elo - away_elo,
        'home_elo_winrate': home_elo_wr,
        'away_elo_winrate': away_elo_wr,
        'home_attack': home_attack,
        'away_attack': away_attack,
        'attack_diff': home_attack - away_attack,
        'home_defense': home_defense,
        'away_defense': away_defense,
        'defense_diff': home_defense - away_defense,
        'home_mid': home_mid,
        'away_mid': away_mid,
        'is_neutral': is_neutral,
        'weight': row['weight'],
        'result': row['result']
    })

features_df = pd.DataFrame(features)
print("V2 Feature matrix shape:", features_df.shape)
features_df.head()

## Save V2 Features

In [ ]:
features_df.to_csv(os.path.join(BASE, 'Feature_Engineering', 'features_v2.csv'), index=False)
print("V2 features saved!")
print("Total features:", len(features_df.columns))
print("Shape:", features_df.shape)